# 04 — SVM grouped validation

本 notebook 使用与 Random Forest 完全相同的 37,724 个像元、557 个
polygon、206 个空间组、`sample_weight` 和五个 outer folds，对线性与 RBF
Support Vector Machine 进行 grouped nested validation、有限调参和最终训练。

SVM 专用方法约束：

- `StandardScaler` 只在每个训练 split 内拟合，验证数据只做 transform。
- 不使用 `SVC(probability=True)`：它会额外进行非 grouped 的概率校准，计算量大，
  且不符合当前空间分组验证设计。
- polygon 预测通过同一 polygon 内各像元的平均 multiclass decision score 得到。
- 使用 LinearSVC 作为 feature-stack screening 模型，再在筛选出的特征组合上比较
  LinearSVC 与 RBF-SVC 候选；完整流程仍在 outer training 内完成。
- 所有耗时 inner fold、候选和 outer fold 均单独 checkpoint，可中断续跑。
- 原始 CSV 和 RF 结果只读；SVM 写入独立结果文件夹。

## 正确打开方式

在 Colab 使用 **File → Open notebook → Upload** 上传整个
`04_SVM_grouped_validation.ipynb`，不要把 notebook JSON 原文粘贴到 Python
单元。请从上到下逐步运行。

默认 `RUN_NAME` 是稳定名称。Colab 中断后保持它不变，从头重新运行；已有
checkpoints 会自动复用。修改参数、特征或方法后必须更换 `RUN_NAME`。

## 流程

1. 读取 RF manifest、固定训练行与 outer folds。
2. 从原始 CSV 分块提取 RF 使用的 37,724 行 predictors。
3. 使用训练 fold 内标准化的 LinearSVC 比较五套 feature stacks。
4. 在每个 outer fold 内重新筛选 feature stack，并调节线性/RBF SVM。
5. 汇总 decision-score polygon predictions 和 OOF metrics。
6. 训练最终 SVM，并与 RF、可选的 XGBoost 做同折比较。

### 0.1 挂载 Google Drive

- **作用：** 访问原始 CSV、RF/XGBoost 结果和新的 SVM 结果目录。
- **输入：** Google 账号授权。
- **输出：** `<DURIAN_DATA_ROOT>/` 可访问。
- **耗时：** 首次需要授权，通常很短。

In [ ]:
from pathlib import Path
import os


def _find_repository_root(start: Path) -> Path:
    current = start.resolve()
    while current.parent != current:
        if (current / "README.md").exists() and (current / "code").exists():
            return current
        current = current.parent
    raise RuntimeError("Run this notebook from within the repository tree.")


REPO_ROOT = _find_repository_root(Path.cwd())
DATA_ROOT = Path(
    os.environ.get("DURIAN_DATA_ROOT", REPO_ROOT / "private_data")
).expanduser().resolve()
REPO_OUTPUT_ROOT = Path(
    os.environ.get("DURIAN_OUTPUT_ROOT", REPO_ROOT / "outputs" / "runs")
).expanduser().resolve()
REPO_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Private data root:", DATA_ROOT)
print("Run output root:", REPO_OUTPUT_ROOT)


### 0.2 导入 Python 库

- **作用：** 加载表格、绘图、分组验证、标准化和 SVM 依赖。
- **输入：** Colab 默认 Python 环境。
- **输出：** 后续单元所需库与版本信息。
- **耗时：** 数秒。

In [ ]:
import gc
import hashlib
import json
import os
import platform
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC

warnings.filterwarnings('once')
warnings.filterwarnings('once', category=ConvergenceWarning)
sns.set_theme(style='whitegrid', context='notebook')

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)

### 0.3 设置输入、RF/XGBoost 和 SVM 输出路径

- **作用：** 锁定正式数据源，并使用稳定 RUN_NAME 支持重启续跑。
- **输入：** RF 结果目录；XGBoost 路径仅用于可选比较。
- **输出：** 路径变量与结果子目录。
- **耗时：** 立即完成。

In [ ]:
DRIVE_ROOT = DATA_ROOT
INPUT_CSV = DRIVE_ROOT / 'Bentong_pixel_samples_2025_v1.csv'
RF_RESULT_DIR = (
    DRIVE_ROOT / 'RF_grouped_validation_20260623_153912_UTC'
)
XGB_RESULT_DIR = (
    DRIVE_ROOT / 'XGBoost_grouped_validation_20260624_run01'
)

RUN_NAME = 'SVM_grouped_validation_20260624_run01'
OUTPUT_DIR = REPO_OUTPUT_ROOT / RUN_NAME

TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
MODEL_DIR = OUTPUT_DIR / 'models'
METADATA_DIR = OUTPUT_DIR / 'metadata'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'

VERIFY_INPUT_SHA256 = True
CSV_CHUNK_SIZE = 100_000
SVC_CACHE_MB = 1200

required_paths = [
    INPUT_CSV,
    RF_RESULT_DIR / 'metadata' / 'model_manifest.json',
    RF_RESULT_DIR / 'metadata' / 'nested_overall_metrics.json',
    RF_RESULT_DIR / 'tables' / 'training_rows_used.csv',
    RF_RESULT_DIR / 'tables' / 'fold_assignments.csv',
    RF_RESULT_DIR / 'tables' / 'oof_polygon_predictions.csv',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Required input files are missing:\n' + '\n'.join(missing_paths)
    )

print('Input CSV:', INPUT_CSV)
print('RF result:', RF_RESULT_DIR)
print('SVM output:', OUTPUT_DIR)
print('Optional XGBoost result exists:', XGB_RESULT_DIR.exists())

## 1. 锁定 RF 数据设计和 SVM 参数

### 1.1 读取 RF manifest 与正式 nested metrics

- **作用：** 继承类别、feature stacks、数据哈希、fold 数和 RF 基准结果。
- **输入：** RF metadata JSON。
- **输出：** `rf_manifest` 和 `rf_nested_metrics`。
- **耗时：** 很短。

In [ ]:
with open(
    RF_RESULT_DIR / 'metadata' / 'model_manifest.json',
    'r',
    encoding='utf-8',
) as file:
    rf_manifest = json.load(file)

with open(
    RF_RESULT_DIR / 'metadata' / 'nested_overall_metrics.json',
    'r',
    encoding='utf-8',
) as file:
    rf_nested_metrics = json.load(file)

print('RF model rows:', rf_manifest['model_rows'])
print('RF samples/groups:',
      rf_manifest['sample_count'], rf_manifest['group_count'])
print('RF polygon Durian F1:', rf_nested_metrics['polygon_durian_f1'])

### 1.2 锁定类别、features 和 grouped CV 参数

- **作用：** 保证 SVM 与 RF 使用完全相同的标签和 predictors 定义。
- **输入：** RF manifest。
- **输出：** 类别映射、五套 feature stacks、outer/inner fold 数。
- **耗时：** 立即完成。

In [ ]:
RANDOM_SEED = int(rf_manifest['random_seed'])
N_SPLITS = int(rf_manifest['n_splits'])
INNER_SPLITS = int(rf_manifest['inner_splits'])
WORKFLOW_VERSION = 'svm_grouped_v1_20260624'
TRAINING_WEIGHT_METHOD = (
    'RF polygon sample_weight multiplied by training-split class '
    'frequency factor, then normalized to mean 1'
)
SCALER_WEIGHT_METHOD = 'RF polygon sample_weight (equal polygon total weight)'

CLASS_TO_ID = {
    str(name): int(class_id)
    for name, class_id in rf_manifest['class_to_id'].items()
}
ID_TO_CLASS = {class_id: name for name, class_id in CLASS_TO_ID.items()}
CLASS_IDS = sorted(ID_TO_CLASS)
CLASS_NAMES = [ID_TO_CLASS[class_id] for class_id in CLASS_IDS]
DURIAN_ID = CLASS_TO_ID['Durian']

FEATURE_SETS = {
    str(name): list(features)
    for name, features in rf_manifest['feature_sets'].items()
}
FULL_BANDS = FEATURE_SETS['FULL']
EXPECTED_MODEL_ROWS = int(rf_manifest['model_rows'])
EXPECTED_SAMPLE_COUNT = int(rf_manifest['sample_count'])
EXPECTED_GROUP_COUNT = int(rf_manifest['group_count'])

print('Classes:', CLASS_TO_ID)
print('Feature sets:', {name: len(v) for name, v in FEATURE_SETS.items()})
print('Outer / inner folds:', N_SPLITS, '/', INNER_SPLITS)

### 1.3 设置 LinearSVC screening 与有限 SVM 候选

- **作用：** 用快速线性 SVM 筛选 feature stack，再比较一个线性和四个 RBF 候选。
- **输入：** 人工锁定的小型参数集。
- **输出：** screening 参数与五个 SVM candidates。
- **耗时：** 立即完成。

In [ ]:
LINEAR_SCREENING_PARAMS = {
    'candidate_id': 'SCREEN_LINEAR',
    'model_type': 'linear_svc',
    'C': 1.0,
    'tol': 1e-4,
    'max_iter': 20_000,
}

SVM_CANDIDATES = [
    {
        'candidate_id': 'L0',
        'model_type': 'linear_svc',
        'C': 1.0,
        'tol': 1e-4,
        'max_iter': 20_000,
    },
    {
        'candidate_id': 'R0',
        'model_type': 'rbf_svc',
        'C': 1.0,
        'gamma': 'scale',
        'tol': 1e-3,
    },
    {
        'candidate_id': 'R1',
        'model_type': 'rbf_svc',
        'C': 10.0,
        'gamma': 'scale',
        'tol': 1e-3,
    },
    {
        'candidate_id': 'R2',
        'model_type': 'rbf_svc',
        'C': 10.0,
        'gamma': 0.01,
        'tol': 1e-3,
    },
    {
        'candidate_id': 'R3',
        'model_type': 'rbf_svc',
        'C': 10.0,
        'gamma': 0.1,
        'tol': 1e-3,
    },
]

print('SVM candidate count:', len(SVM_CANDIDATES))
display(pd.DataFrame(SVM_CANDIDATES))

### 1.4 创建可续跑结果目录与 run signature

- **作用：** 阻止不同代码版本、数据或参数误用旧 checkpoints。
- **输入：** RF 文件哈希、sklearn 版本与 SVM 配置。
- **输出：** 独立输出目录和 `run_config.json`。
- **耗时：** 很短。

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


training_rows_path = RF_RESULT_DIR / 'tables' / 'training_rows_used.csv'
fold_assignments_path = RF_RESULT_DIR / 'tables' / 'fold_assignments.csv'

run_signature_payload = {
    'workflow_version': WORKFLOW_VERSION,
    'scikit_learn_version': sklearn.__version__,
    'source_rf_manifest_sha256': sha256_file(
        RF_RESULT_DIR / 'metadata' / 'model_manifest.json'
    ),
    'training_rows_sha256': sha256_file(training_rows_path),
    'fold_assignments_sha256': sha256_file(fold_assignments_path),
    'random_seed': RANDOM_SEED,
    'n_splits': N_SPLITS,
    'inner_splits': INNER_SPLITS,
    'feature_sets': FEATURE_SETS,
    'linear_screening_params': LINEAR_SCREENING_PARAMS,
    'svm_candidates': SVM_CANDIDATES,
    'svc_cache_mb': SVC_CACHE_MB,
    'training_weight_method': TRAINING_WEIGHT_METHOD,
    'scaler_weight_method': SCALER_WEIGHT_METHOD,
    'probability': False,
    'polygon_aggregation': 'mean multiclass decision score',
}
run_signature = hashlib.sha256(
    json.dumps(
        run_signature_payload,
        sort_keys=True,
        ensure_ascii=False,
    ).encode('utf-8')
).hexdigest()

for folder in [
    OUTPUT_DIR, TABLE_DIR, FIGURE_DIR,
    MODEL_DIR, METADATA_DIR, CHECKPOINT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

run_config_path = METADATA_DIR / 'run_config.json'
if run_config_path.exists():
    with open(run_config_path, 'r', encoding='utf-8') as file:
        existing_config = json.load(file)
    if existing_config.get('run_signature') != run_signature:
        raise ValueError(
            'RUN_NAME contains incompatible checkpoints. '
            'Change RUN_NAME before continuing.'
        )
    print('Compatible existing run found; checkpoints may be reused.')
else:
    with open(run_config_path, 'w', encoding='utf-8') as file:
        json.dump(
            {
                'created_utc': datetime.now(timezone.utc).isoformat(),
                'run_name': RUN_NAME,
                'run_signature': run_signature,
                **run_signature_payload,
            },
            file,
            indent=2,
            ensure_ascii=False,
        )

print('Run signature:', run_signature)
print('Output directory:', OUTPUT_DIR)

## 2. 复用 RF 训练行并低内存读取 predictors

### 2.1 读取 RF 训练行和 fold assignments

- **作用：** 复用完全相同的像元、polygon 权重和 outer folds。
- **输入：** RF 两个 CSV。
- **输出：** `training_rows`、`fold_assignments`。
- **耗时：** 很短。

In [ ]:
training_rows = pd.read_csv(
    training_rows_path,
    dtype={
        'pixel_uid': 'string',
        'sample_uid': 'string',
        'group_uid': 'string',
        'class_lv2': 'string',
    },
)
fold_assignments = pd.read_csv(
    fold_assignments_path,
    dtype={
        'sample_uid': 'string',
        'group_uid': 'string',
        'class_lv2': 'string',
    },
)
for column in ['class_id', 'fold_id']:
    training_rows[column] = pd.to_numeric(
        training_rows[column], errors='raise'
    ).astype(int)
    fold_assignments[column] = pd.to_numeric(
        fold_assignments[column], errors='raise'
    ).astype(int)
training_rows['sample_weight'] = pd.to_numeric(
    training_rows['sample_weight'], errors='raise'
).astype(float)

print('Training rows:', f'{len(training_rows):,}')
print('Fold assignments:', len(fold_assignments))
display(training_rows.head())

### 2.2 审计像元、权重和空间隔离

- **作用：** 确认 37,724/557/206、pixel_uid 唯一、polygon 总权重为 1、group 不跨 fold。
- **输入：** 复用的 RF 训练设计。
- **输出：** 审计表与 `training_rows_reused.csv`。
- **耗时：** 较短。

In [ ]:
if len(training_rows) != EXPECTED_MODEL_ROWS:
    raise ValueError('Unexpected reused training row count.')
if training_rows['pixel_uid'].isna().any():
    raise ValueError('pixel_uid contains null values.')
if training_rows['pixel_uid'].duplicated().any():
    raise ValueError('pixel_uid is not unique.')
if training_rows['sample_uid'].nunique() != EXPECTED_SAMPLE_COUNT:
    raise ValueError('Unexpected polygon count.')
if training_rows['group_uid'].nunique() != EXPECTED_GROUP_COUNT:
    raise ValueError('Unexpected group count.')
if sorted(training_rows['class_id'].unique()) != CLASS_IDS:
    raise ValueError('Not all locked classes are present.')
if sorted(training_rows['fold_id'].unique()) != list(range(N_SPLITS)):
    raise ValueError('Unexpected outer fold IDs.')
if training_rows.groupby('group_uid')['fold_id'].nunique().max() != 1:
    raise ValueError('A group_uid crosses outer folds.')

polygon_weight_totals = training_rows.groupby('sample_uid')[
    'sample_weight'
].sum()
if not np.allclose(polygon_weight_totals.to_numpy(), 1.0):
    raise ValueError('Polygon sample weights do not sum to 1.')

sample_design = training_rows[
    ['sample_uid', 'group_uid', 'class_id', 'class_lv2', 'fold_id']
].drop_duplicates('sample_uid')
fold_check = sample_design.merge(
    fold_assignments,
    on='sample_uid',
    how='outer',
    suffixes=('_training', '_assignment'),
    indicator=True,
    validate='one_to_one',
)
if not (fold_check['_merge'] == 'both').all():
    raise ValueError('RF training rows and fold assignments differ.')
for column in ['group_uid', 'class_id', 'class_lv2', 'fold_id']:
    if not (
        fold_check[f'{column}_training'].astype(str)
        == fold_check[f'{column}_assignment'].astype(str)
    ).all():
        raise ValueError(f'Fold metadata mismatch: {column}')

reused_summary = (
    training_rows.groupby(['class_id', 'class_lv2'])
    .agg(
        pixel_rows=('pixel_uid', 'size'),
        samples=('sample_uid', 'nunique'),
        groups=('group_uid', 'nunique'),
    )
    .reset_index()
    .sort_values('class_id')
)
training_rows.to_csv(
    TABLE_DIR / 'training_rows_reused.csv', index=False
)
reused_summary.to_csv(
    TABLE_DIR / 'reused_training_summary.csv', index=False
)
display(reused_summary)

### 2.3 验证原始 CSV SHA256

- **作用：** 确认 SVM 与 RF 使用同一个原始 CSV。
- **输入：** 原始 CSV 与 RF manifest 哈希。
- **输出：** `actual_input_sha256`。
- **耗时：** 顺序读取文件，通常几十秒至数分钟。

In [ ]:
if VERIFY_INPUT_SHA256:
    hash_start = time.time()
    actual_input_sha256 = sha256_file(INPUT_CSV)
    print(
        f'CSV SHA256 runtime: {(time.time() - hash_start) / 60:.1f} minutes'
    )
    if actual_input_sha256 != rf_manifest['input_csv_sha256']:
        raise ValueError('Input CSV SHA256 differs from RF manifest.')
else:
    actual_input_sha256 = rf_manifest['input_csv_sha256']
    print('SHA256 verification skipped by configuration.')
print('Input SHA256:', actual_input_sha256)

### 2.4 分块提取固定 pixel_uid 的 26 个 predictors

- **作用：** 避免把 712,772 行 CSV 一次载入内存。
- **输入：** 原始 CSV 与 37,724 个 RF pixel_uid。
- **输出：** `feature_rows`。
- **耗时：** 数十秒至数分钟。

In [ ]:
selected_pixel_ids = set(training_rows['pixel_uid'].astype(str))
feature_parts = []
scanned_rows = 0
retained_rows = 0

for chunk_index, chunk in enumerate(
    pd.read_csv(
        INPUT_CSV,
        usecols=['pixel_uid'] + FULL_BANDS,
        dtype={'pixel_uid': 'string'},
        chunksize=CSV_CHUNK_SIZE,
        low_memory=False,
    ),
    start=1,
):
    scanned_rows += len(chunk)
    keep_mask = chunk['pixel_uid'].astype(str).isin(selected_pixel_ids)
    retained_chunk = chunk.loc[keep_mask].copy()
    retained_rows += len(retained_chunk)
    if not retained_chunk.empty:
        feature_parts.append(retained_chunk)
    print(
        f'Chunk {chunk_index}: scanned={scanned_rows:,}, '
        f'retained={retained_rows:,}'
    )
    del chunk, retained_chunk
    gc.collect()

if not feature_parts:
    raise ValueError('No selected pixel_uid values were found.')
feature_rows = pd.concat(feature_parts, ignore_index=True)
feature_parts.clear()
gc.collect()

if feature_rows['pixel_uid'].duplicated().any():
    raise ValueError('Selected features contain duplicate pixel_uid.')
if len(feature_rows) != EXPECTED_MODEL_ROWS:
    raise ValueError(
        f'Expected {EXPECTED_MODEL_ROWS} features, found '
        f'{len(feature_rows)}.'
    )
print('Selected feature rows:', f'{len(feature_rows):,}')

### 2.5 合并训练设计与 predictors

- **作用：** 建立唯一 SVM 建模表，并将 predictors 转成 float32。
- **输入：** `training_rows`、`feature_rows`。
- **输出：** `model_df`、`sample_table` 和数据审计 JSON。
- **耗时：** 较短。

In [ ]:
model_df = training_rows.merge(
    feature_rows,
    on='pixel_uid',
    how='left',
    validate='one_to_one',
)
del feature_rows
gc.collect()

for feature in FULL_BANDS:
    model_df[feature] = pd.to_numeric(
        model_df[feature], errors='coerce'
    ).astype('float32')
model_df[FULL_BANDS] = model_df[FULL_BANDS].replace(
    [np.inf, -np.inf], np.nan
)
if model_df[FULL_BANDS].isna().any(axis=1).any():
    raise ValueError('At least one reused pixel has a missing predictor.')

sample_table = (
    model_df[
        ['sample_uid', 'group_uid', 'class_id', 'class_lv2', 'fold_id']
    ]
    .drop_duplicates('sample_uid')
    .sort_values('sample_uid')
    .reset_index(drop=True)
)
data_audit = {
    'input_csv_rows_scanned': int(scanned_rows),
    'model_rows': int(len(model_df)),
    'sample_count': int(model_df['sample_uid'].nunique()),
    'group_count': int(model_df['group_uid'].nunique()),
    'predictor_count': len(FULL_BANDS),
    'memory_mb': float(model_df.memory_usage(deep=True).sum() / 1e6),
}
with open(
    METADATA_DIR / 'svm_data_audit.json', 'w', encoding='utf-8'
) as file:
    json.dump(data_audit, file, indent=2, ensure_ascii=False)
print(json.dumps(data_audit, indent=2))

## 3. SVM、decision scores、metrics 与 checkpoint 函数

### 3.1 定义原子写入函数

- **作用：** 避免中断后留下不完整的正式 JSON/CSV。
- **输入：** Python 对象或 dataframe。
- **输出：** checkpoint I/O 工具。
- **耗时：** 只定义函数。

In [ ]:
def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f'Not JSON serializable: {type(value)}')


def atomic_write_json(payload, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + '.tmp')
    with open(temporary_path, 'w', encoding='utf-8') as file:
        json.dump(
            payload,
            file,
            indent=2,
            ensure_ascii=False,
            default=json_default,
        )
    os.replace(temporary_path, output_path)


def atomic_write_csv(frame, output_path, index=False):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + '.tmp')
    frame.to_csv(temporary_path, index=index)
    os.replace(temporary_path, output_path)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as file:
        return json.load(file)

### 3.2 定义训练权重与无泄露 StandardScaler–SVM pipeline

- **作用：** 在每个训练 split 内拟合 scaler，并将 polygon/类别平衡权重传给分类器。
- **输入：** 训练 dataframe、features 和候选参数。
- **输出：** 已拟合 Pipeline。
- **耗时：** 只定义函数。

In [ ]:
def make_svm_training_weights(train_frame):
    class_row_counts = train_frame['class_id'].value_counts()
    if set(class_row_counts.index.astype(int)) != set(CLASS_IDS):
        raise ValueError('A training split is missing a class.')
    class_factors = {
        int(class_id): len(train_frame)
        / (len(CLASS_IDS) * int(row_count))
        for class_id, row_count in class_row_counts.items()
    }
    weights = (
        train_frame['sample_weight'].to_numpy(dtype='float64')
        * train_frame['class_id'].map(class_factors).to_numpy(dtype='float64')
    )
    weights *= len(weights) / weights.sum()
    return weights.astype('float64')


def clean_svm_params(params):
    return {
        key: value
        for key, value in params.items()
        if key != 'candidate_id'
    }


def build_svm_pipeline(params):
    model_type = params['model_type']
    if model_type == 'linear_svc':
        classifier = LinearSVC(
            C=float(params['C']),
            tol=float(params.get('tol', 1e-4)),
            max_iter=int(params.get('max_iter', 20_000)),
            dual='auto',
            class_weight=None,
            random_state=RANDOM_SEED,
        )
    elif model_type == 'rbf_svc':
        classifier = SVC(
            C=float(params['C']),
            kernel='rbf',
            gamma=params['gamma'],
            tol=float(params.get('tol', 1e-3)),
            probability=False,
            class_weight=None,
            cache_size=SVC_CACHE_MB,
            shrinking=True,
            decision_function_shape='ovr',
            break_ties=True,
            max_iter=-1,
            random_state=RANDOM_SEED,
        )
    else:
        raise ValueError(f'Unknown model_type: {model_type}')

    return Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', classifier),
    ])


def fit_svm(train_frame, features, params):
    model = build_svm_pipeline(params)
    model.fit(
        train_frame[features],
        train_frame['class_id'],
        scaler__sample_weight=train_frame[
            'sample_weight'
        ].to_numpy(dtype='float64'),
        classifier__sample_weight=make_svm_training_weights(train_frame),
    )
    classifier = model.named_steps['classifier']
    if isinstance(classifier, LinearSVC):
        observed_iterations = int(
            np.max(np.atleast_1d(classifier.n_iter_))
        )
        configured_max_iter = int(params.get('max_iter', 20_000))
        if observed_iterations >= configured_max_iter:
            warnings.warn(
                'LinearSVC reached max_iter; review convergence before '
                'using this configuration.',
                RuntimeWarning,
            )
    return model

### 3.3 定义像元 decision score 与 polygon 聚合

- **作用：** 不做非 grouped 概率校准；对每类 decision score 在 polygon 内取平均。
- **输入：** 已拟合 pipeline 和验证数据。
- **输出：** 像元与 polygon predictions。
- **耗时：** 只定义函数。

In [ ]:
def make_pixel_predictions(model, frame, features):
    predicted_ids = model.predict(frame[features]).astype(int)
    decision_scores = model.decision_function(frame[features])
    classifier = model.named_steps['classifier']
    model_classes = np.asarray(classifier.classes_).astype(int)

    if decision_scores.ndim != 2 or decision_scores.shape[1] != len(
        model_classes
    ):
        raise ValueError('Unexpected multiclass decision score shape.')

    output = frame[[
        'pixel_uid', 'sample_uid', 'group_uid', 'class_lv2',
        'class_id', 'fold_id', 'sample_weight',
    ]].copy()
    output['pred_class_id'] = predicted_ids
    output['pred_class_name'] = output['pred_class_id'].map(ID_TO_CLASS)

    for class_id in CLASS_IDS:
        output[f'score_{class_id}'] = np.nan
    for column_index, class_id in enumerate(model_classes):
        output[f'score_{class_id}'] = decision_scores[:, column_index]
    if output[[f'score_{class_id}' for class_id in CLASS_IDS]].isna().any().any():
        raise ValueError('At least one class decision score is missing.')
    return output


def aggregate_polygon_predictions(pixel_predictions):
    score_columns = [f'score_{class_id}' for class_id in CLASS_IDS]
    aggregations = {
        'group_uid': 'first',
        'class_lv2': 'first',
        'class_id': 'first',
        'fold_id': 'first',
    }
    aggregations.update({column: 'mean' for column in score_columns})
    polygon_predictions = (
        pixel_predictions.groupby('sample_uid', as_index=False)
        .agg(aggregations)
    )
    score_matrix = polygon_predictions[score_columns].to_numpy()
    polygon_predictions['pred_class_id'] = np.asarray(CLASS_IDS)[
        score_matrix.argmax(axis=1)
    ]
    polygon_predictions['pred_class_name'] = polygon_predictions[
        'pred_class_id'
    ].map(ID_TO_CLASS)
    return polygon_predictions

### 3.4 定义与 RF/XGBoost 相同的 metrics

- **作用：** 计算等 polygon 权重像元指标与 polygon 指标。
- **输入：** 真实/预测类别和可选权重。
- **输出：** accuracy、balanced accuracy、macro F1、Durian metrics。
- **耗时：** 只定义函数。

In [ ]:
def metric_dictionary(y_true, y_pred, sample_weight=None):
    precision, recall, durian_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[DURIAN_ID],
        average=None,
        sample_weight=sample_weight,
        zero_division=0,
    )
    return {
        'accuracy': accuracy_score(
            y_true, y_pred, sample_weight=sample_weight
        ),
        'balanced_accuracy': balanced_accuracy_score(
            y_true, y_pred, sample_weight=sample_weight
        ),
        'macro_f1': f1_score(
            y_true,
            y_pred,
            labels=CLASS_IDS,
            average='macro',
            sample_weight=sample_weight,
            zero_division=0,
        ),
        'durian_precision': float(precision[0]),
        'durian_recall': float(recall[0]),
        'durian_f1': float(durian_f1[0]),
    }


def evaluate_model(model, validation_frame, features):
    pixel_predictions = make_pixel_predictions(
        model, validation_frame, features
    )
    polygon_predictions = aggregate_polygon_predictions(pixel_predictions)
    pixel_metrics = metric_dictionary(
        pixel_predictions['class_id'],
        pixel_predictions['pred_class_id'],
        sample_weight=pixel_predictions['sample_weight'],
    )
    polygon_metrics = metric_dictionary(
        polygon_predictions['class_id'],
        polygon_predictions['pred_class_id'],
    )
    return (
        {
            **{
                f'pixel_{key}': value
                for key, value in pixel_metrics.items()
            },
            **{
                f'polygon_{key}': value
                for key, value in polygon_metrics.items()
            },
        },
        pixel_predictions,
        polygon_predictions,
    )

### 3.5 定义报告和混淆矩阵输出

- **作用：** 统一保存七类报告、CSV 和 PNG。
- **输入：** OOF predictions。
- **输出：** 分类报告和两套 confusion matrices。
- **耗时：** 只定义函数。

In [ ]:
def report_frame(y_true, y_pred, weights=None):
    return pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=CLASS_IDS,
            target_names=CLASS_NAMES,
            sample_weight=weights,
            zero_division=0,
            output_dict=True,
        )
    ).T


def save_confusion_figure(
    y_true, y_pred, title, output_name, weights=None
):
    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=CLASS_IDS,
        sample_weight=weights,
    )
    matrix_frame = pd.DataFrame(
        matrix, index=CLASS_NAMES, columns=CLASS_NAMES
    )
    atomic_write_csv(
        matrix_frame,
        TABLE_DIR / f'{output_name}.csv',
        index=True,
    )
    plt.figure(figsize=(9, 7))
    sns.heatmap(matrix_frame, annot=True, fmt='.1f', cmap='Purples')
    plt.title(title)
    plt.xlabel('Predicted class')
    plt.ylabel('Reference class')
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / f'{output_name}.png',
        dpi=180,
        bbox_inches='tight',
    )
    plt.show()

### 3.6 定义有效 inner grouped folds

- **作用：** 寻找每个 inner fold 均包含七类且 group 不跨 fold 的可复用方案。
- **输入：** outer training polygon 表。
- **输出：** inner fold assignments。
- **耗时：** 只定义函数。

In [ ]:
def build_valid_grouped_assignments(
    samples, n_splits, base_seed, max_attempts=1000
):
    samples = samples.reset_index(drop=True).copy()
    dummy_x = np.zeros((len(samples), 1))
    best_fold_ids = None
    best_seed = None
    best_score = np.inf

    for attempt in range(max_attempts):
        seed = base_seed + attempt
        splitter = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=seed,
        )
        fold_ids = np.full(len(samples), -1, dtype=int)
        for fold_id, (_, validation_indices) in enumerate(
            splitter.split(
                dummy_x,
                samples['class_id'],
                groups=samples['group_uid'],
            )
        ):
            fold_ids[validation_indices] = fold_id

        test_frame = samples.assign(inner_fold_id=fold_ids)
        class_counts = pd.crosstab(
            test_frame['inner_fold_id'], test_frame['class_id']
        ).reindex(
            index=range(n_splits), columns=CLASS_IDS, fill_value=0
        )
        if (class_counts == 0).any().any():
            continue
        group_counts = (
            test_frame.groupby(['inner_fold_id', 'class_id'])[
                'group_uid'
            ]
            .nunique()
            .unstack(fill_value=0)
            .reindex(
                index=range(n_splits),
                columns=CLASS_IDS,
                fill_value=0,
            )
        )
        expected = 1.0 / n_splits
        class_proportions = class_counts.div(
            class_counts.sum(axis=0), axis=1
        )
        group_proportions = group_counts.div(
            group_counts.sum(axis=0), axis=1
        )
        score = (
            ((class_proportions - expected) ** 2).to_numpy().sum()
            + ((group_proportions - expected) ** 2).to_numpy().sum()
        )
        if score < best_score:
            best_fold_ids = fold_ids.copy()
            best_seed = seed
            best_score = score

    if best_fold_ids is None:
        raise ValueError('Could not create valid inner grouped folds.')

    assignments = samples[
        ['sample_uid', 'group_uid', 'class_id', 'class_lv2']
    ].copy()
    assignments['inner_fold_id'] = best_fold_ids
    assignments['split_seed'] = best_seed
    if assignments.groupby('group_uid')['inner_fold_id'].nunique().max() != 1:
        raise AssertionError('An inner group crosses folds.')
    return assignments, best_seed, best_score


def get_or_create_inner_assignments(outer_fold_id, outer_train_samples):
    path = (
        CHECKPOINT_DIR / 'nested' / f'outer_{outer_fold_id}'
        / 'inner_fold_assignments.csv'
    )
    if path.exists():
        assignments = pd.read_csv(
            path,
            dtype={
                'sample_uid': 'string',
                'group_uid': 'string',
                'class_lv2': 'string',
            },
        )
        if set(assignments['sample_uid'].astype(str)) != set(
            outer_train_samples['sample_uid'].astype(str)
        ):
            raise ValueError('Stale inner fold assignments.')
        return assignments

    assignments, split_seed, balance_score = (
        build_valid_grouped_assignments(
            outer_train_samples,
            n_splits=INNER_SPLITS,
            base_seed=RANDOM_SEED + 2000 + outer_fold_id * 1000,
        )
    )
    atomic_write_csv(assignments, path, index=False)
    print(
        f'Outer {outer_fold_id}: inner seed={split_seed}, '
        f'balance={balance_score:.6f}'
    )
    return assignments

### 3.7 定义逐 inner-fold checkpoint 的配置评分

- **作用：** 一个慢速 RBF inner fold 完成后立即保存，断线时不需要重做。
- **输入：** outer training、inner assignments、features 和 SVM params。
- **输出：** inner metrics 与平均分数。
- **耗时：** 只定义函数。

In [ ]:
SELECTION_METRICS = [
    'polygon_durian_f1_mean',
    'polygon_macro_f1_mean',
    'pixel_durian_f1_mean',
]


def score_configuration_checkpointed(
    outer_train_pixels,
    inner_assignments,
    features,
    params,
    checkpoint_directory,
    checkpoint_prefix,
):
    checkpoint_directory = Path(checkpoint_directory)
    checkpoint_directory.mkdir(parents=True, exist_ok=True)
    fold_records = []

    for inner_fold_id in range(INNER_SPLITS):
        checkpoint_path = (
            checkpoint_directory
            / f'{checkpoint_prefix}_inner_{inner_fold_id}.json'
        )
        if checkpoint_path.exists():
            record = load_json(checkpoint_path)
            print(f'    inner {inner_fold_id}: reused')
            fold_records.append(record)
            continue

        validation_uids = set(
            inner_assignments.loc[
                inner_assignments['inner_fold_id'] == inner_fold_id,
                'sample_uid',
            ].astype(str)
        )
        train_frame = outer_train_pixels.loc[
            ~outer_train_pixels['sample_uid'].astype(str).isin(
                validation_uids
            )
        ]
        validation_frame = outer_train_pixels.loc[
            outer_train_pixels['sample_uid'].astype(str).isin(
                validation_uids
            )
        ]
        if set(train_frame['group_uid']) & set(validation_frame['group_uid']):
            raise AssertionError('Inner group leakage.')

        fit_start = time.time()
        model = fit_svm(train_frame, features, params)
        metrics, pixel_predictions, polygon_predictions = evaluate_model(
            model, validation_frame, features
        )
        record = {
            'inner_fold_id': inner_fold_id,
            'runtime_minutes': (time.time() - fit_start) / 60,
            **metrics,
        }
        atomic_write_json(record, checkpoint_path)
        fold_records.append(record)
        del model, pixel_predictions, polygon_predictions
        gc.collect()
        print(
            f'    inner {inner_fold_id}: completed in '
            f'{record["runtime_minutes"]:.1f} min'
        )

    fold_frame = pd.DataFrame(fold_records)
    return {
        'polygon_durian_f1_mean': float(
            fold_frame['polygon_durian_f1'].mean()
        ),
        'polygon_macro_f1_mean': float(
            fold_frame['polygon_macro_f1'].mean()
        ),
        'pixel_durian_f1_mean': float(
            fold_frame['pixel_durian_f1'].mean()
        ),
        'pixel_macro_f1_mean': float(
            fold_frame['pixel_macro_f1'].mean()
        ),
        'runtime_minutes_sum': float(
            fold_frame['runtime_minutes'].sum()
        ),
    }

## 4. LinearSVC feature-stack screening

### 4.1 使用固定 LinearSVC 比较五套 feature stacks

- **作用：** 快速比较 data-source contribution；每个 feature × outer fold 单独 checkpoint。
- **输入：** RF outer folds 与五套 features。
- **输出：** 25 个 screening fold scores。
- **耗时：** 中等；LinearSVC 通常明显快于 RBF。

In [ ]:
feature_checkpoint_dir = CHECKPOINT_DIR / 'feature_screening'
feature_checkpoint_dir.mkdir(parents=True, exist_ok=True)
feature_fold_records = []

for feature_set_name, features in FEATURE_SETS.items():
    print(f'Feature set: {feature_set_name} ({len(features)} features)')
    for fold_id in range(N_SPLITS):
        checkpoint_path = (
            feature_checkpoint_dir
            / f'{feature_set_name}_fold_{fold_id}.json'
        )
        if checkpoint_path.exists():
            record = load_json(checkpoint_path)
            print(f'  fold {fold_id}: reused')
        else:
            train_frame = model_df.loc[model_df['fold_id'] != fold_id]
            validation_frame = model_df.loc[
                model_df['fold_id'] == fold_id
            ]
            fit_start = time.time()
            model = fit_svm(
                train_frame,
                features,
                LINEAR_SCREENING_PARAMS,
            )
            metrics, pixel_predictions, polygon_predictions = evaluate_model(
                model, validation_frame, features
            )
            record = {
                'feature_set': feature_set_name,
                'feature_count': len(features),
                'fold_id': fold_id,
                'runtime_minutes': (time.time() - fit_start) / 60,
                **metrics,
            }
            atomic_write_json(record, checkpoint_path)
            del model, pixel_predictions, polygon_predictions
            gc.collect()
            print(f'  fold {fold_id}: completed')
        feature_fold_records.append(record)

feature_fold_metrics = pd.DataFrame(feature_fold_records)
atomic_write_csv(
    feature_fold_metrics,
    TABLE_DIR / 'feature_set_fold_metrics.csv',
    index=False,
)

### 4.2 汇总 full-data screening 结果

- **作用：** 按 polygon Durian F1、polygon macro F1、pixel Durian F1 排序。
- **输入：** LinearSVC screening fold metrics。
- **输出：** `SELECTED_FEATURE_SET`、CSV 与比较图。
- **耗时：** 较短。

In [ ]:
feature_summary = (
    feature_fold_metrics.groupby('feature_set')
    .agg(
        feature_count=('feature_count', 'first'),
        polygon_durian_f1_mean=('polygon_durian_f1', 'mean'),
        polygon_durian_f1_std=('polygon_durian_f1', 'std'),
        polygon_macro_f1_mean=('polygon_macro_f1', 'mean'),
        polygon_macro_f1_std=('polygon_macro_f1', 'std'),
        pixel_durian_f1_mean=('pixel_durian_f1', 'mean'),
        pixel_macro_f1_mean=('pixel_macro_f1', 'mean'),
        runtime_minutes_sum=('runtime_minutes', 'sum'),
    )
    .reset_index()
    .sort_values(SELECTION_METRICS, ascending=False)
    .reset_index(drop=True)
)
SELECTED_FEATURE_SET = str(feature_summary.loc[0, 'feature_set'])
SELECTED_FEATURES = FEATURE_SETS[SELECTED_FEATURE_SET]
atomic_write_csv(
    feature_summary,
    TABLE_DIR / 'feature_set_summary.csv',
    index=False,
)

plot_data = feature_summary.melt(
    id_vars='feature_set',
    value_vars=[
        'polygon_durian_f1_mean', 'polygon_macro_f1_mean'
    ],
    var_name='metric',
    value_name='score',
)
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_data, x='feature_set', y='score', hue='metric')
plt.ylim(0, 1)
plt.title('LinearSVC feature-stack screening')
plt.xlabel('Feature set')
plt.ylabel('Mean 5-fold score')
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / 'feature_set_comparison.png',
    dpi=180,
    bbox_inches='tight',
)
plt.show()

print('Selected feature set:', SELECTED_FEATURE_SET)
display(feature_summary)

## 5. Nested grouped feature screening 与 SVM 调参

### 5.1 运行逐 inner-fold checkpoint 的 nested SVM

- **作用：** 每个 outer training 内以 LinearSVC 选 features，再比较线性/RBF candidates。
- **输入：** 固定 outer folds、有效 inner folds、features 和 candidates。
- **输出：** nested checkpoints、outer OOF predictions 与 metrics。
- **耗时：** 全 notebook 最耗时；每个 inner fit 均可续跑。

In [ ]:
nested_feature_records = []
nested_candidate_records = []
nested_outer_records = []
nested_selected_records = []

nested_start = time.time()
for outer_fold_id in range(N_SPLITS):
    print(f'\n===== Outer fold {outer_fold_id + 1}/{N_SPLITS} =====')
    outer_dir = CHECKPOINT_DIR / 'nested' / f'outer_{outer_fold_id}'
    outer_dir.mkdir(parents=True, exist_ok=True)

    outer_train_pixels = model_df.loc[
        model_df['fold_id'] != outer_fold_id
    ]
    outer_validation_pixels = model_df.loc[
        model_df['fold_id'] == outer_fold_id
    ]
    outer_train_samples = sample_table.loc[
        sample_table['fold_id'] != outer_fold_id
    ]
    if set(outer_train_pixels['group_uid']) & set(
        outer_validation_pixels['group_uid']
    ):
        raise AssertionError('Outer group leakage.')
    inner_assignments = get_or_create_inner_assignments(
        outer_fold_id, outer_train_samples
    )

    # Stage A: LinearSVC feature screening inside outer training.
    fold_feature_rows = []
    for feature_set_name, features in FEATURE_SETS.items():
        print(f'  screening feature: {feature_set_name}')
        scores = score_configuration_checkpointed(
            outer_train_pixels,
            inner_assignments,
            features,
            LINEAR_SCREENING_PARAMS,
            checkpoint_directory=outer_dir / 'feature_scores',
            checkpoint_prefix=f'feature_{feature_set_name}',
        )
        row = {
            'outer_fold_id': outer_fold_id,
            'feature_set': feature_set_name,
            'feature_count': len(features),
            **scores,
        }
        fold_feature_rows.append(row)
        nested_feature_records.append(row)

    fold_feature_frame = (
        pd.DataFrame(fold_feature_rows)
        .sort_values(SELECTION_METRICS, ascending=False)
        .reset_index(drop=True)
    )
    fold_feature_set = str(fold_feature_frame.loc[0, 'feature_set'])
    fold_features = FEATURE_SETS[fold_feature_set]
    print('  selected feature set:', fold_feature_set)

    # Stage B: tune linear/RBF SVM on the fold-specific feature stack.
    fold_candidate_rows = []
    for candidate in SVM_CANDIDATES:
        candidate_id = candidate['candidate_id']
        print(f'  tuning candidate: {candidate_id}')
        scores = score_configuration_checkpointed(
            outer_train_pixels,
            inner_assignments,
            fold_features,
            candidate,
            checkpoint_directory=outer_dir / 'candidate_scores',
            checkpoint_prefix=f'candidate_{candidate_id}',
        )
        row = {
            'outer_fold_id': outer_fold_id,
            'selected_feature_set': fold_feature_set,
            'candidate_id': candidate_id,
            **scores,
        }
        fold_candidate_rows.append(row)
        nested_candidate_records.append(row)

    fold_candidate_frame = (
        pd.DataFrame(fold_candidate_rows)
        .sort_values(SELECTION_METRICS, ascending=False)
        .reset_index(drop=True)
    )
    best_candidate_id = str(
        fold_candidate_frame.loc[0, 'candidate_id']
    )
    best_candidate = next(
        candidate
        for candidate in SVM_CANDIDATES
        if candidate['candidate_id'] == best_candidate_id
    )
    print('  selected candidate:', best_candidate_id)
    nested_selected_records.append({
        'outer_fold_id': outer_fold_id,
        'selected_feature_set': fold_feature_set,
        'selected_feature_count': len(fold_features),
        **best_candidate,
    })

    outer_metrics_path = outer_dir / 'outer_metrics.json'
    outer_pixel_path = outer_dir / 'oof_pixel_predictions.csv'
    outer_polygon_path = outer_dir / 'oof_polygon_predictions.csv'
    complete_path = outer_dir / 'complete.json'

    if (
        complete_path.exists()
        and outer_metrics_path.exists()
        and outer_pixel_path.exists()
        and outer_polygon_path.exists()
    ):
        outer_record = load_json(outer_metrics_path)
        print('  outer evaluation: reused')
    else:
        fit_start = time.time()
        outer_model = fit_svm(
            outer_train_pixels, fold_features, best_candidate
        )
        metrics, pixel_predictions, polygon_predictions = evaluate_model(
            outer_model, outer_validation_pixels, fold_features
        )
        outer_record = {
            'outer_fold_id': outer_fold_id,
            'selected_feature_set': fold_feature_set,
            'candidate_id': best_candidate_id,
            'runtime_minutes': (time.time() - fit_start) / 60,
            **metrics,
        }
        atomic_write_csv(pixel_predictions, outer_pixel_path, index=False)
        atomic_write_csv(
            polygon_predictions, outer_polygon_path, index=False
        )
        atomic_write_json(outer_record, outer_metrics_path)
        atomic_write_json(
            {
                'complete': True,
                'completed_utc': datetime.now(timezone.utc).isoformat(),
                'run_signature': run_signature,
            },
            complete_path,
        )
        del outer_model, pixel_predictions, polygon_predictions
        gc.collect()
        print('  outer evaluation: completed')

    nested_outer_records.append(outer_record)
    del outer_train_pixels, outer_validation_pixels, inner_assignments
    gc.collect()

print(
    f'Nested runtime this session: '
    f'{(time.time() - nested_start) / 60:.1f} minutes'
)

nested_feature_scores = pd.DataFrame(nested_feature_records)
nested_candidate_scores = pd.DataFrame(nested_candidate_records)
nested_fold_metrics = pd.DataFrame(nested_outer_records)
nested_selected = pd.DataFrame(nested_selected_records)
atomic_write_csv(
    nested_feature_scores,
    TABLE_DIR / 'nested_inner_feature_set_scores.csv',
    index=False,
)
atomic_write_csv(
    nested_candidate_scores,
    TABLE_DIR / 'nested_inner_candidate_scores.csv',
    index=False,
)
atomic_write_csv(
    nested_fold_metrics,
    TABLE_DIR / 'nested_outer_fold_metrics.csv',
    index=False,
)
atomic_write_csv(
    nested_selected,
    TABLE_DIR / 'nested_selected_candidates.csv',
    index=False,
)
display(nested_fold_metrics)

### 5.2 低内存合并 outer-fold OOF 文件

- **作用：** 逐文件写出完整 OOF CSV，仅载入 metrics 所需字段。
- **输入：** 五个 outer fold predictions。
- **输出：** 完整像元/polygon OOF CSV 与轻量评价表。
- **耗时：** 较短，不使用大规模 concat。

In [ ]:
def stream_csv_files(input_paths, output_path, chunksize=50_000):
    output_path = Path(output_path)
    temporary_path = output_path.with_suffix(output_path.suffix + '.tmp')
    if temporary_path.exists():
        temporary_path.unlink()
    total_rows = 0
    for input_path in input_paths:
        for chunk in pd.read_csv(input_path, chunksize=chunksize):
            chunk.to_csv(
                temporary_path,
                mode='w' if total_rows == 0 else 'a',
                header=total_rows == 0,
                index=False,
            )
            total_rows += len(chunk)
            del chunk
            gc.collect()
    if total_rows == 0:
        raise ValueError(f'No rows written to {output_path}.')
    os.replace(temporary_path, output_path)
    return total_rows


outer_dirs = [
    CHECKPOINT_DIR / 'nested' / f'outer_{fold_id}'
    for fold_id in range(N_SPLITS)
]
pixel_fold_paths = [
    folder / 'oof_pixel_predictions.csv' for folder in outer_dirs
]
polygon_fold_paths = [
    folder / 'oof_polygon_predictions.csv' for folder in outer_dirs
]
missing_oof = [
    str(path)
    for path in pixel_fold_paths + polygon_fold_paths
    if not path.exists()
]
if missing_oof:
    raise FileNotFoundError('Missing OOF files:\n' + '\n'.join(missing_oof))

oof_pixel_path = TABLE_DIR / 'oof_pixel_predictions.csv'
oof_polygon_path = TABLE_DIR / 'oof_polygon_predictions.csv'
pixel_rows_written = stream_csv_files(pixel_fold_paths, oof_pixel_path)
polygon_rows_written = stream_csv_files(
    polygon_fold_paths, oof_polygon_path
)

oof_pixel_predictions = pd.read_csv(
    oof_pixel_path,
    usecols=[
        'pixel_uid', 'sample_uid', 'class_id',
        'pred_class_id', 'sample_weight',
    ],
    dtype={
        'pixel_uid': 'string',
        'sample_uid': 'string',
        'class_id': 'int16',
        'pred_class_id': 'int16',
        'sample_weight': 'float32',
    },
)
oof_polygon_predictions = pd.read_csv(
    oof_polygon_path,
    dtype={
        'sample_uid': 'string',
        'group_uid': 'string',
        'class_lv2': 'string',
        'class_id': 'int16',
        'pred_class_id': 'int16',
    },
)
if pixel_rows_written != EXPECTED_MODEL_ROWS:
    raise ValueError('Unexpected OOF pixel row count.')
if polygon_rows_written != EXPECTED_SAMPLE_COUNT:
    raise ValueError('Unexpected OOF polygon row count.')
if oof_pixel_predictions['pixel_uid'].duplicated().any():
    raise ValueError('Duplicate OOF pixel_uid values.')
if oof_polygon_predictions['sample_uid'].duplicated().any():
    raise ValueError('Duplicate OOF sample_uid values.')

print('OOF pixel rows:', f'{pixel_rows_written:,}')
print('OOF polygon rows:', f'{polygon_rows_written:,}')

### 5.3 生成 nested OOF metrics、报告和混淆矩阵

- **作用：** 得到正式的无空间组泄露内部验证结果。
- **输入：** OOF predictions。
- **输出：** metrics JSON、classification reports 和 confusion matrices。
- **耗时：** 数秒至数十秒。

In [ ]:
overall_pixel_metrics = metric_dictionary(
    oof_pixel_predictions['class_id'],
    oof_pixel_predictions['pred_class_id'],
    sample_weight=oof_pixel_predictions['sample_weight'],
)
overall_polygon_metrics = metric_dictionary(
    oof_polygon_predictions['class_id'],
    oof_polygon_predictions['pred_class_id'],
)
nested_feature_counts = {
    str(name): int(count)
    for name, count in nested_selected[
        'selected_feature_set'
    ].value_counts().items()
}
nested_candidate_counts = {
    str(name): int(count)
    for name, count in nested_selected[
        'candidate_id'
    ].value_counts().items()
}
nested_overall_metrics = {
    'full_data_selected_feature_set': SELECTED_FEATURE_SET,
    'outer_fold_selected_feature_set_counts': nested_feature_counts,
    'outer_fold_selected_candidate_counts': nested_candidate_counts,
    **{
        f'pixel_{key}': value
        for key, value in overall_pixel_metrics.items()
    },
    **{
        f'polygon_{key}': value
        for key, value in overall_polygon_metrics.items()
    },
}
atomic_write_json(
    nested_overall_metrics,
    METADATA_DIR / 'nested_overall_metrics.json',
)

pixel_report = report_frame(
    oof_pixel_predictions['class_id'],
    oof_pixel_predictions['pred_class_id'],
    weights=oof_pixel_predictions['sample_weight'],
)
polygon_report = report_frame(
    oof_polygon_predictions['class_id'],
    oof_polygon_predictions['pred_class_id'],
)
atomic_write_csv(
    pixel_report,
    TABLE_DIR / 'oof_pixel_classification_report.csv',
    index=True,
)
atomic_write_csv(
    polygon_report,
    TABLE_DIR / 'oof_polygon_classification_report.csv',
    index=True,
)
save_confusion_figure(
    oof_pixel_predictions['class_id'],
    oof_pixel_predictions['pred_class_id'],
    'SVM nested grouped CV — weighted pixel confusion matrix',
    'oof_pixel_confusion_matrix',
    weights=oof_pixel_predictions['sample_weight'],
)
save_confusion_figure(
    oof_polygon_predictions['class_id'],
    oof_polygon_predictions['pred_class_id'],
    'SVM nested grouped CV — polygon confusion matrix',
    'oof_polygon_confusion_matrix',
)
print(json.dumps(nested_overall_metrics, indent=2))
display(polygon_report)

## 6. 选择最终 SVM 并使用全部 Bentong 数据训练

### 6.1 在固定 outer folds 上评价最终 candidates

- **作用：** nested 性能估计结束后，为最终部署模型选定一组参数；每个 candidate × fold checkpoint。
- **输入：** full-data selected features、SVM candidates 和 RF outer folds。
- **输出：** 25 个 final candidate fold scores。
- **耗时：** 较耗时，RBF fits 可续跑。

In [ ]:
final_checkpoint_dir = CHECKPOINT_DIR / 'final_candidate_selection'
final_checkpoint_dir.mkdir(parents=True, exist_ok=True)
final_fold_records = []

for candidate in SVM_CANDIDATES:
    candidate_id = candidate['candidate_id']
    print(f'Final candidate: {candidate_id}')
    for fold_id in range(N_SPLITS):
        checkpoint_path = (
            final_checkpoint_dir
            / f'{candidate_id}_fold_{fold_id}.json'
        )
        if checkpoint_path.exists():
            record = load_json(checkpoint_path)
            print(f'  fold {fold_id}: reused')
        else:
            train_frame = model_df.loc[model_df['fold_id'] != fold_id]
            validation_frame = model_df.loc[
                model_df['fold_id'] == fold_id
            ]
            fit_start = time.time()
            model = fit_svm(
                train_frame, SELECTED_FEATURES, candidate
            )
            metrics, pixel_predictions, polygon_predictions = evaluate_model(
                model, validation_frame, SELECTED_FEATURES
            )
            record = {
                'candidate_id': candidate_id,
                'fold_id': fold_id,
                'runtime_minutes': (time.time() - fit_start) / 60,
                **metrics,
            }
            atomic_write_json(record, checkpoint_path)
            del model, pixel_predictions, polygon_predictions
            gc.collect()
            print(f'  fold {fold_id}: completed')
        final_fold_records.append(record)

final_candidate_fold_metrics = pd.DataFrame(final_fold_records)
atomic_write_csv(
    final_candidate_fold_metrics,
    TABLE_DIR / 'final_candidate_fold_metrics.csv',
    index=False,
)

### 6.2 汇总并选择最终 candidate

- **作用：** 沿用 polygon Durian F1、polygon macro F1、pixel Durian F1 排序。
- **输入：** candidate-fold metrics。
- **输出：** `FINAL_SVM_PARAMS` 和 summary CSV。
- **耗时：** 较短。

In [ ]:
final_candidate_scores = (
    final_candidate_fold_metrics.groupby('candidate_id')
    .agg(
        polygon_durian_f1_mean=('polygon_durian_f1', 'mean'),
        polygon_macro_f1_mean=('polygon_macro_f1', 'mean'),
        pixel_durian_f1_mean=('pixel_durian_f1', 'mean'),
        pixel_macro_f1_mean=('pixel_macro_f1', 'mean'),
        runtime_minutes_sum=('runtime_minutes', 'sum'),
    )
    .reset_index()
    .sort_values(SELECTION_METRICS, ascending=False)
    .reset_index(drop=True)
)
FINAL_CANDIDATE_ID = str(
    final_candidate_scores.loc[0, 'candidate_id']
)
FINAL_SVM_PARAMS = next(
    candidate
    for candidate in SVM_CANDIDATES
    if candidate['candidate_id'] == FINAL_CANDIDATE_ID
)
atomic_write_csv(
    final_candidate_scores,
    TABLE_DIR / 'final_candidate_scores.csv',
    index=False,
)
print('Selected feature set:', SELECTED_FEATURE_SET)
print('Selected candidate:', FINAL_CANDIDATE_ID)
display(final_candidate_scores)

### 6.3 训练并保存最终 StandardScaler–SVM pipeline

- **作用：** 拟合全部 Bentong 像元；若 joblib 已存在则直接载入。
- **输入：** 最终 features、candidate 和全部建模行。
- **输出：** 最终 model、bundle 和 selected features。
- **耗时：** 首次运行中等至较长，取决于是否选择 RBF。

In [ ]:
final_model_path = MODEL_DIR / 'svm_final_model.joblib'
if final_model_path.exists():
    final_model = joblib.load(final_model_path)
    print('Existing final SVM loaded.')
else:
    fit_start = time.time()
    final_model = fit_svm(
        model_df, SELECTED_FEATURES, FINAL_SVM_PARAMS
    )
    joblib.dump(final_model, final_model_path)
    print(
        f'Final SVM trained in '
        f'{(time.time() - fit_start) / 60:.1f} minutes.'
    )

model_bundle = {
    'model': final_model,
    'feature_set': SELECTED_FEATURE_SET,
    'predictor_bands': SELECTED_FEATURES,
    'class_to_id': CLASS_TO_ID,
    'id_to_class': ID_TO_CLASS,
    'svm_params': FINAL_SVM_PARAMS,
    'candidate_id': FINAL_CANDIDATE_ID,
    'random_seed': RANDOM_SEED,
    'decision_score_aggregation': 'mean per polygon',
    'probability_calibration': False,
    'source_rf_result_dir': str(RF_RESULT_DIR),
    'run_signature': run_signature,
}
joblib.dump(model_bundle, MODEL_DIR / 'svm_final_bundle.joblib')
atomic_write_json(
    {'selected_predictor_bands': SELECTED_FEATURES},
    METADATA_DIR / 'selected_predictors.json',
)

### 6.4 输出最终模型 diagnostics

- **作用：** LinearSVC 输出绝对系数；RBF-SVC 输出各类 support-vector 数量，并明确无内置 feature importance。
- **输入：** 最终 pipeline。
- **输出：** diagnostics JSON 和可用时的 coefficient CSV/PNG。
- **耗时：** 较短。

In [ ]:
final_classifier = final_model.named_steps['classifier']
model_diagnostics = {
    'candidate_id': FINAL_CANDIDATE_ID,
    'model_type': FINAL_SVM_PARAMS['model_type'],
    'probability': False,
    'decision_function_shape': 'ovr',
    'has_intrinsic_feature_importance': False,
}

if isinstance(final_classifier, LinearSVC):
    coefficient_importance = np.abs(final_classifier.coef_).mean(axis=0)
    coefficient_frame = (
        pd.DataFrame({
            'feature': SELECTED_FEATURES,
            'mean_absolute_scaled_coefficient': coefficient_importance,
        })
        .sort_values(
            'mean_absolute_scaled_coefficient', ascending=False
        )
        .reset_index(drop=True)
    )
    atomic_write_csv(
        coefficient_frame,
        TABLE_DIR / 'linear_svm_coefficient_importance.csv',
        index=False,
    )
    plt.figure(figsize=(9, max(5, len(coefficient_frame) * 0.28)))
    sns.barplot(
        data=coefficient_frame,
        y='feature',
        x='mean_absolute_scaled_coefficient',
        color='#8B5CF6',
    )
    plt.title('Linear SVM mean absolute scaled coefficient')
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / 'linear_svm_coefficient_importance.png',
        dpi=180,
        bbox_inches='tight',
    )
    plt.show()
    model_diagnostics['has_intrinsic_feature_importance'] = True
elif isinstance(final_classifier, SVC):
    model_diagnostics['n_support_by_class'] = {
        ID_TO_CLASS[int(class_id)]: int(count)
        for class_id, count in zip(
            final_classifier.classes_, final_classifier.n_support_
        )
    }
    model_diagnostics['total_support_vectors'] = int(
        final_classifier.n_support_.sum()
    )
    model_diagnostics['importance_note'] = (
        'RBF-SVC has no direct coefficient-based feature importance.'
    )

atomic_write_json(
    model_diagnostics,
    METADATA_DIR / 'model_diagnostics.json',
)
print(json.dumps(model_diagnostics, indent=2))

## 7. 与 RF 和可用的 XGBoost 做同折比较

### 7.1 生成统一 metrics 和 polygon 配对比较

- **作用：** 比较相同 outer folds、相同 557 个 polygon；若 XGBoost 已完成则自动加入。
- **输入：** RF/SVM metrics 和可选 XGBoost metrics。
- **输出：** 模型比较 CSV、polygon 正确性表和图。
- **耗时：** 较短。

In [ ]:
comparison_metric_names = [
    'pixel_accuracy',
    'pixel_balanced_accuracy',
    'pixel_macro_f1',
    'pixel_durian_f1',
    'polygon_accuracy',
    'polygon_balanced_accuracy',
    'polygon_macro_f1',
    'polygon_durian_f1',
]
comparison_rows = [
    {
        'model': 'Random Forest',
        **{
            metric: rf_nested_metrics[metric]
            for metric in comparison_metric_names
        },
    },
    {
        'model': 'SVM',
        **{
            metric: nested_overall_metrics[metric]
            for metric in comparison_metric_names
        },
    },
]

xgb_metrics_path = XGB_RESULT_DIR / 'metadata' / 'nested_overall_metrics.json'
xgb_polygon_path = XGB_RESULT_DIR / 'tables' / 'oof_polygon_predictions.csv'
xgb_metrics = None
if xgb_metrics_path.exists():
    xgb_metrics = load_json(xgb_metrics_path)
    comparison_rows.append({
        'model': 'XGBoost',
        **{
            metric: xgb_metrics[metric]
            for metric in comparison_metric_names
        },
    })
    print('XGBoost metrics included.')
else:
    print('XGBoost result not found; RF–SVM comparison created.')

model_comparison = pd.DataFrame(comparison_rows)
atomic_write_csv(
    model_comparison,
    TABLE_DIR / 'model_nested_metrics_comparison.csv',
    index=False,
)

rf_polygon = pd.read_csv(
    RF_RESULT_DIR / 'tables' / 'oof_polygon_predictions.csv',
    usecols=['sample_uid', 'class_id', 'pred_class_id'],
    dtype={
        'sample_uid': 'string',
        'class_id': 'int16',
        'pred_class_id': 'int16',
    },
).rename(columns={'pred_class_id': 'rf_pred_class_id'})
svm_polygon = oof_polygon_predictions[
    ['sample_uid', 'class_id', 'pred_class_id']
].rename(columns={'pred_class_id': 'svm_pred_class_id'})

paired = rf_polygon.merge(
    svm_polygon,
    on='sample_uid',
    how='outer',
    suffixes=('_rf', '_svm'),
    indicator=True,
    validate='one_to_one',
)
if not (paired['_merge'] == 'both').all():
    raise ValueError('RF and SVM OOF sample sets differ.')
if not (paired['class_id_rf'] == paired['class_id_svm']).all():
    raise ValueError('RF and SVM reference classes differ.')
paired['class_id'] = paired['class_id_rf']
paired['rf_correct'] = (
    paired['rf_pred_class_id'] == paired['class_id']
)
paired['svm_correct'] = (
    paired['svm_pred_class_id'] == paired['class_id']
)
paired['correctness_pattern'] = (
    paired['rf_correct'].astype(int).astype(str)
    + '_'
    + paired['svm_correct'].astype(int).astype(str)
)

if xgb_polygon_path.exists():
    xgb_polygon = pd.read_csv(
        xgb_polygon_path,
        usecols=['sample_uid', 'pred_class_id'],
        dtype={
            'sample_uid': 'string',
            'pred_class_id': 'int16',
        },
    ).rename(columns={'pred_class_id': 'xgb_pred_class_id'})
    paired = paired.merge(
        xgb_polygon,
        on='sample_uid',
        how='left',
        validate='one_to_one',
    )
    paired['xgb_correct'] = (
        paired['xgb_pred_class_id'] == paired['class_id']
    )

atomic_write_csv(
    paired.drop(columns=['_merge']),
    TABLE_DIR / 'paired_polygon_model_predictions.csv',
    index=False,
)
disagreement_summary = (
    paired.groupby('correctness_pattern')
    .size()
    .rename('polygon_count')
    .reset_index()
)
atomic_write_csv(
    disagreement_summary,
    TABLE_DIR / 'rf_svm_correctness_disagreement.csv',
    index=False,
)

plot_metrics = model_comparison.melt(
    id_vars='model',
    value_vars=[
        'polygon_accuracy',
        'polygon_macro_f1',
        'polygon_durian_f1',
    ],
    var_name='metric',
    value_name='score',
)
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_metrics, x='metric', y='score', hue='model')
plt.ylim(0, 1)
plt.title('Nested grouped CV model comparison')
plt.xlabel('')
plt.ylabel('Score')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / 'model_nested_metric_comparison.png',
    dpi=180,
    bbox_inches='tight',
)
plt.show()

display(model_comparison)
display(disagreement_summary)

## 8. 保存 manifest、README 和 inventory

### 8.1 写入最终可复现记录

- **作用：** 记录数据哈希、RF 来源、SVM 设计、版本、selected features、参数和 metrics。
- **输入：** 本次完整运行状态。
- **输出：** manifest、README、inventory 和 completion marker。
- **耗时：** 较短。

In [ ]:
manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'run_name': RUN_NAME,
    'run_signature': run_signature,
    'workflow_version': WORKFLOW_VERSION,
    'input_csv': str(INPUT_CSV),
    'input_csv_sha256': actual_input_sha256,
    'source_rf_result_directory': str(RF_RESULT_DIR),
    'source_rf_manifest_sha256': run_signature_payload[
        'source_rf_manifest_sha256'
    ],
    'source_training_rows_sha256': run_signature_payload[
        'training_rows_sha256'
    ],
    'source_fold_assignments_sha256': run_signature_payload[
        'fold_assignments_sha256'
    ],
    'output_directory': str(OUTPUT_DIR),
    'model_rows': int(len(model_df)),
    'sample_count': int(model_df['sample_uid'].nunique()),
    'group_count': int(model_df['group_uid'].nunique()),
    'n_splits': N_SPLITS,
    'inner_splits': INNER_SPLITS,
    'random_seed': RANDOM_SEED,
    'svc_cache_mb': SVC_CACHE_MB,
    'class_to_id': CLASS_TO_ID,
    'feature_sets': FEATURE_SETS,
    'feature_screening_model': LINEAR_SCREENING_PARAMS,
    'selected_feature_set': SELECTED_FEATURE_SET,
    'selected_predictor_bands': SELECTED_FEATURES,
    'final_candidate_id': FINAL_CANDIDATE_ID,
    'final_svm_params': FINAL_SVM_PARAMS,
    'standardization': 'StandardScaler fitted inside every training split',
    'scaler_weight_method': SCALER_WEIGHT_METHOD,
    'probability_calibration': False,
    'polygon_aggregation': 'mean multiclass decision score',
    'training_weight_method': TRAINING_WEIGHT_METHOD,
    'nested_overall_metrics': nested_overall_metrics,
    'rf_nested_overall_metrics': rf_nested_metrics,
    'xgboost_nested_overall_metrics': xgb_metrics,
    'model_diagnostics': model_diagnostics,
    'software': {
        'python': sys.version,
        'platform': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'joblib': joblib.__version__,
    },
}
atomic_write_json(manifest, METADATA_DIR / 'model_manifest.json')

readme_text = f'''SVM grouped validation results
==============================
Input CSV: {INPUT_CSV}
Source RF result: {RF_RESULT_DIR}
Reused model rows: {len(model_df)}
Samples: {model_df['sample_uid'].nunique()}
Groups: {model_df['group_uid'].nunique()}
Selected feature set: {SELECTED_FEATURE_SET}
Selected candidate: {FINAL_CANDIDATE_ID}
Predictor count: {len(SELECTED_FEATURES)}

Method notes
------------
StandardScaler is fitted only on each training split.
LinearSVC screens feature stacks; linear/RBF candidates are then tuned.
SVC probability calibration is disabled.
Polygon predictions use mean multiclass decision score.
Checkpoints permit resume with the same RUN_NAME.

Key files
---------
models/svm_final_model.joblib
models/svm_final_bundle.joblib
metadata/model_manifest.json
metadata/nested_overall_metrics.json
metadata/model_diagnostics.json
tables/feature_set_summary.csv
tables/nested_outer_fold_metrics.csv
tables/oof_pixel_predictions.csv
tables/oof_polygon_predictions.csv
tables/final_candidate_scores.csv
tables/model_nested_metrics_comparison.csv
figures/oof_pixel_confusion_matrix.png
figures/oof_polygon_confusion_matrix.png
figures/model_nested_metric_comparison.png
'''
(OUTPUT_DIR / 'README.txt').write_text(readme_text, encoding='utf-8')

inventory = []
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        inventory.append({
            'relative_path': str(path.relative_to(OUTPUT_DIR)),
            'size_bytes': int(path.stat().st_size),
        })
inventory_frame = pd.DataFrame(inventory)
atomic_write_csv(
    inventory_frame,
    TABLE_DIR / 'output_inventory.csv',
    index=False,
)
atomic_write_json(
    {
        'complete': True,
        'completed_utc': datetime.now(timezone.utc).isoformat(),
        'run_signature': run_signature,
    },
    METADATA_DIR / 'RUN_COMPLETE.json',
)

print('SVM workflow completed successfully.')
print('Results folder:', OUTPUT_DIR)
display(inventory_frame)